In [1]:
#%%capture
%run input/Format.ipynb
import ROOT as root
from array import array
root.gErrorIgnoreLevel = root.kFatal
%jsroot on

/home/yoren/.local/lib/python3.10/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Welcome to JupyROOT 6.30/06


In [2]:
iOption0 = 0
Save_to_Html = False
Save_to_pdf = False

In [3]:
N_centr = 10
N_real_centr = 5
eIDbins = [1,2,3,4,5]
primaryID = [[i for i in range(1,26)],[19,20,24,25],[20,25]]
colors=[1,2,4,root.kGreen+2,root.kMagenta,root.kOrange+4,root.kGray,root.kCyan]
central_bins = [0,20,20,40,40,60,60,80,80,93]
real_file_path = 'input/' 
real_file_names=['my-m_ee_Run14AuAu_55th_new_19657_1072runs.root','../../VTXalignment/input/my-m_ee_Run14AuAu_143rd_new_20013_1065runs.root']
#m_ee_Run14AuAu_49th_new_19636_1072runs m_ee_Run14AuAu_50th_new_19640_1072runs my-m_ee_Run14AuAu_54th_new_19654_1072runs m_ee_Run14AuAu_55th_new_19657_1072runs
#m_ee_Run14AuAu_45th_new_19617_240runs , m_ee_Run14AuAu_44th_new_19614_975runs, m_ee_Run14AuAu_46th_new_19619_999runs m_ee_Run14AuAu_47th_new_19623_1078runs
sim_file_path="../sim/output/Newembed/sngl/"
primary_file_names=['phi_50M_v6.root']
hadron_file_names=['charged_pi_100M_v3.root']
dalitz_file_names=['dalitz_50M_v7.root']
conv_file_names=['conv_100M.root']
sim_file_names = [primary_file_names,hadron_file_names,dalitz_file_names,conv_file_names ]
sim_file_names = [dalitz_file_names, hadron_file_names, primary_file_names ]
part_names = ["c#bar{c}","dalitz","#gamma #rightarrow ee","#pi^{-}"]
eID_names = ["loose","tight","BDT loose","BDT basic","BDT tight"]
hist_names = ["el_pt_hist_"]
hist_pt_orig_name = "hist_pt_orig"

In [4]:
hists_read_real, hists_read_sim = [], []

for iFilereal in range(len(real_file_names)):
    infile = root.TFile.Open(real_file_path+real_file_names[iFilereal], "read")
    hists_read_real_ifile = []
    for icentr in range(N_centr):
        hists_read_real_ifile.append(infile.Get(hist_names[0]+f'{icentr}'))
        hists_read_real_ifile[-1].SetDirectory(root.nullptr)
        hists_read_real_ifile[-1].SetName(hist_names[0]+f'{icentr}'+'_real')
    hists_read_real.append(hists_read_real_ifile)
    infile.Close()

for iFilesim in range(len(sim_file_names)):
    infile = root.TFile.Open(sim_file_path+sim_file_names[iFilesim][0], "read")
    hists_read_sim_ifile = []
    for icentr in range(N_centr):
        hists_read_sim_ifile.append(infile.Get(hist_names[0]+f'{icentr}'))
        hists_read_sim_ifile[-1].SetDirectory(root.nullptr)
        hists_read_sim_ifile[-1].SetName(hist_names[0]+f'{icentr}'+f'_sim{iFilesim}_')
    hists_read_sim.append(hists_read_sim_ifile)
    if iFilesim == 0:   
        hist_pt_orig = infile.Get(hist_pt_orig_name)
        hist_pt_orig.SetDirectory(root.nullptr)
    infile.Close()

In [5]:
hits_sim_new_names = [
    "e_pt_hist_AI", "h_pt_hist_AI", "e_pt_hist_SC", "h_pt_hist_SC", "e_pt_hist_NC", "h_pt_hist_NC"
]
new_sim_file = root.TFile.Open("../ML/output/BDT_test1.root", "read")#output/BDT_test1.root
new_sim_hists = []
for ihist in range(len(hits_sim_new_names)):
    new_sim_hists.append(new_sim_file.Get(hits_sim_new_names[ihist]))
    new_sim_hists[-1].SetDirectory(root.nullptr)
new_sim_file.Close()

In [6]:
hists_pt_real, hists_pt_sim = [], []

for iFilereal in range(len(real_file_names)):
    hists_pt_real_eid = []
    for ieID in range(len(eIDbins)):
        hists_pt_real_conv = []
        for iconv in range(len(primaryID)):
            hists_pt_real_centr = []
            for icentr in range(N_centr):
                hists_pt_real_centr.append(hists_read_real[0][icentr].ProjectionX(hists_read_real[iFilereal][icentr].GetName()+f"_pr{ieID}_{iconv}_0",eIDbins[ieID],eIDbins[ieID],primaryID[iconv][0],primaryID[iconv][0]))
                for jconv in range(1,len(primaryID[iconv])):
                    hists_pt_real_centr[-1].Add(hists_read_real[0][icentr].ProjectionX(hists_read_real[iFilereal][icentr].GetName()+f"_pr{ieID}_{iconv}_{jconv}",eIDbins[ieID],eIDbins[ieID],primaryID[iconv][jconv],primaryID[iconv][jconv]))
            hists_pt_real_conv.append(hists_pt_real_centr)
        hists_pt_real_eid.append(hists_pt_real_conv)
    hists_pt_real.append(hists_pt_real_eid)
    

for iFilesim in range(len(sim_file_names)):
    hists_pt_sim_eid = []
    for ieID in range(len(eIDbins)):
        hists_pt_sim_conv = []
        for iconv in range(len(primaryID)):
            hists_pt_sim_centr = []
            for icentr in range(N_centr):
                hists_pt_sim_centr.append(hists_read_sim[iFilesim][icentr].ProjectionX(hists_read_sim[iFilesim][icentr].GetName()+f"_pr{ieID}_{iconv}_0",eIDbins[ieID],eIDbins[ieID],primaryID[iconv][0],primaryID[iconv][0]))
                for jconv in range(1,len(primaryID[iconv])):
                    hists_pt_sim_centr[-1].Add(hists_read_sim[iFilesim][icentr].ProjectionX(hists_read_sim[iFilesim][icentr].GetName()+f"_pr{ieID}_{iconv}_{jconv}",eIDbins[ieID],eIDbins[ieID],primaryID[iconv][jconv],primaryID[iconv][jconv]))
            hists_pt_sim_conv.append(hists_pt_sim_centr)
        hists_pt_sim_eid.append(hists_pt_sim_conv)
    hists_pt_sim.append(hists_pt_sim_eid)

In [7]:
import random

In [8]:
c0 = root.TCanvas(f"c0",f"c0",900,900)
c0.Divide(1)
h = []
legends = []
iED = [0,4]
for icentr in range(1):
    c0.cd(icentr+1)
    h.append(root.TH1D(f"name{icentr}{ieID}{iconv}", f"name{icentr}{ieID}{iconv}", 100, 0.0, 5))
    h[-1].SetMaximum(1.2* max(hists_pt_real[0][iED[0]][0][icentr].GetMaximum(),hists_pt_real[0][iED[1]][0][icentr+5].GetMaximum()))
    h[-1].SetMinimum(10)
    Format_Hist_total(h[-1], title_x="p_{T} [GeV]",  title_y="dN/dp_{T}", left=0.15, bottom=0.15, right=0.01, top=0.01,  Tsize=0.07,  Lsize=0.06,\
                          Mstyle=21,  Msize=0, Mcolor=1,  Lwidth=6,  Lcolor=1,  offset_x=1, offset_y=1, title="",  Malpha=1,  Lalpha=1)
    
    Format_Hist_total(hists_pt_real[0][iED[0]][0][icentr], title_x="p_{T} [GeV]",  title_y="dN/dp_{T}", left=0.15, bottom=0.15, right=0.01, top=0.01,  Tsize=0.07,  Lsize=0.06,\
              Mstyle=21,  Msize=0, Mcolor=1,  Lwidth=7,  Lcolor=root.kRed+2,  offset_x=1, offset_y=1, title="",  Malpha=1,  Lalpha=1)
    Format_Hist_total(hists_pt_real[1][iED[1]][0][icentr], title_x="p_{T} [GeV]",  title_y="dN/dp_{T}", left=0.15, bottom=0.15, right=0.01, top=0.01,  Tsize=0.07,  Lsize=0.06,\
              Mstyle=21,  Msize=0, Mcolor=4,  Lwidth=7,  Lcolor=root.kBlue+2,  offset_x=1, offset_y=1, title="",  Malpha=1,  Lalpha=1)
    
    root.gPad.SetLogy()
    h[-1].Draw()
    
    #scale_data = 2.268594e9*10
    bdt_sim = new_sim_hists[0].ProjectionX(f"bdt_sim{icentr}",icentr+1,icentr+2)
    bdt_had = new_sim_hists[1].ProjectionX(f"bdt_had{icentr}",icentr+1,icentr+2)
    bdt_had.Smooth(1)
    std_sim = new_sim_hists[2].ProjectionX(f"std_sim{icentr}",icentr+1,icentr+2)
    std_had = new_sim_hists[3].ProjectionX(f"std_had{icentr}",icentr+1,icentr+2)

    rescale1, rescale2 = [], []

    for ibin in range(27,1+bdt_had.GetNbinsX()): bdt_had.SetBinContent(ibin, bdt_had.GetBinContent(ibin)/2)
    for ibin in range(1,1+bdt_sim.GetNbinsX()):
        if bdt_sim.GetBinContent(ibin) > 0:
            rescale1.append(hists_pt_real[1][iED[1]][0][icentr].GetBinContent(ibin)/(bdt_sim.GetBinContent(ibin)+bdt_had.GetBinContent(ibin)))
        else:
            rescale1.append(1)
        if std_sim.GetBinContent(ibin) > 0:
            rescale2.append(hists_pt_real[0][iED[0]][0][icentr].GetBinContent(ibin)/(std_sim.GetBinContent(ibin)+std_had.GetBinContent(ibin)))
        else:
            rescale2.append(1)
    for ibin in range(1,1+bdt_sim.GetNbinsX()):
        if len(rescale1) > 0:
            bdt_sim.SetBinContent(ibin, bdt_sim.GetBinContent(ibin)*rescale1[ibin-1])
            bdt_had.SetBinContent(ibin, bdt_had.GetBinContent(ibin)*rescale1[ibin-1])
        if len(rescale2) > 0:
            std_sim.SetBinContent(ibin, std_sim.GetBinContent(ibin)*rescale2[ibin-1])
            std_had.SetBinContent(ibin, std_had.GetBinContent(ibin)*rescale2[ibin-1])


    hists_pt_real[0][iED[0]][0][icentr].Draw("same")
    hists_pt_real[1][iED[1]][0][icentr].Draw("same")
    Format_Graph(bdt_sim, Lcolor = root.kBlue, Lwidth=3)
    Format_Graph(bdt_had, Lcolor = root.kBlue-7, Lwidth=3)
    Format_Graph(std_sim, Lcolor = root.kRed, Lwidth=3)
    Format_Graph(std_had, Lcolor = root.kRed-7, Lwidth=3)
    bdt_sim.Draw("same HIST")
    bdt_had.Draw("same HIST")
    std_sim.Draw("same HIST")
    std_had.Draw("same HIST")
    
    legends.append(root.TLegend(0.4,0.88,0.99,0.99,"0-20% Au+Au 200 GeV"))
    Format_Legend(legends[-1])
    legends[-1].Draw()
    legends.append(root.TLegend(0.4,0.84,0.99,0.92,""))
    legends[-1].SetNColumns(2)
    legends[-1].AddEntry(hists_pt_real[0][iED[0]][0][icentr],"data std","l")
    legends[-1].AddEntry(hists_pt_real[0][iED[1]][0][icentr],"data bdt","l")
    Format_Legend(legends[-1])
    legends[-1].Draw()
    legends.append(root.TLegend(0.4,0.78,0.99,0.86,""))
    legends[-1].SetNColumns(2)
    legends[-1].AddEntry(std_sim,"e sim std","l")
    legends[-1].AddEntry(bdt_sim,"e sim bdt","l")
    Format_Legend(legends[-1])
    legends[-1].Draw()
    legends.append(root.TLegend(0.4,0.72,0.99,0.80,""))
    legends[-1].SetNColumns(2)
    legends[-1].AddEntry(std_had,"h sim std","l")
    legends[-1].AddEntry(bdt_had,"h sim bdt","l")
    Format_Legend(legends[-1])
    legends[-1].Draw()
if not Save_to_Html: c0.Draw()
c0.SaveAs("output/electron_hadron_decomposition.png")

In [9]:
c0 = root.TCanvas(f"c0",f"c0",1800,1200)
c0.Divide(3,2)
h = []
legends = []
iED = [0,4]
for icentr in range(5):
    c0.cd(icentr+1)
    h.append(root.TH1D(f"name{icentr}{ieID}{iconv}", f"name{icentr}{ieID}{iconv}", 100, 0.0, 5))
    h[-1].SetMaximum(1.2* max(hists_pt_real[0][iED[0]][0][icentr].GetMaximum(),hists_pt_real[0][iED[1]][0][icentr+5].GetMaximum()))
    h[-1].SetMinimum(10)
    Format_Hist_total(h[-1], title_x="p_{T} [GeV]",  title_y="dN/dp_{T}", left=0.15, bottom=0.15, right=0.01, top=0.01,  Tsize=0.07,  Lsize=0.06,\
                          Mstyle=21,  Msize=0, Mcolor=1,  Lwidth=6,  Lcolor=1,  offset_x=1, offset_y=1, title="",  Malpha=1,  Lalpha=1)
    
    Format_Hist_total(hists_pt_real[0][iED[0]][0][icentr], title_x="p_{T} [GeV]",  title_y="dN/dp_{T}", left=0.15, bottom=0.15, right=0.01, top=0.01,  Tsize=0.07,  Lsize=0.06,\
              Mstyle=21,  Msize=0, Mcolor=1,  Lwidth=7,  Lcolor=root.kRed+2,  offset_x=1, offset_y=1, title="",  Malpha=1,  Lalpha=1)
    Format_Hist_total(hists_pt_real[1][iED[1]][0][icentr], title_x="p_{T} [GeV]",  title_y="dN/dp_{T}", left=0.15, bottom=0.15, right=0.01, top=0.01,  Tsize=0.07,  Lsize=0.06,\
              Mstyle=21,  Msize=0, Mcolor=4,  Lwidth=7,  Lcolor=root.kBlue+2,  offset_x=1, offset_y=1, title="",  Malpha=1,  Lalpha=1)
    
    root.gPad.SetLogy()
    h[-1].Draw()
    
    #scale_data = 2.268594e9*10
    bdt_sim = new_sim_hists[0].ProjectionX(f"bdt_sim{icentr}",icentr+1,icentr+2)
    bdt_had = new_sim_hists[1].ProjectionX(f"bdt_had{icentr}",icentr+1,icentr+2)
    bdt_had.Smooth(1)
    std_sim = new_sim_hists[2].ProjectionX(f"std_sim{icentr}",icentr+1,icentr+2)
    std_had = new_sim_hists[3].ProjectionX(f"std_had{icentr}",icentr+1,icentr+2)

    rescale1, rescale2 = [], []

    for ibin in range(27,1+bdt_had.GetNbinsX()): bdt_had.SetBinContent(ibin, bdt_had.GetBinContent(ibin)/2)
    for ibin in range(1,1+bdt_sim.GetNbinsX()):
        if bdt_sim.GetBinContent(ibin) > 0:
            rescale1.append(hists_pt_real[1][iED[1]][0][icentr].GetBinContent(ibin)/(bdt_sim.GetBinContent(ibin)+bdt_had.GetBinContent(ibin)))
        else:
            rescale1.append(1)
        if std_sim.GetBinContent(ibin) > 0:
            rescale2.append(hists_pt_real[0][iED[0]][0][icentr].GetBinContent(ibin)/(std_sim.GetBinContent(ibin)+std_had.GetBinContent(ibin)))
        else:
            rescale2.append(1)
    for ibin in range(1,1+bdt_sim.GetNbinsX()):
        if len(rescale1) > 0:
            bdt_sim.SetBinContent(ibin, bdt_sim.GetBinContent(ibin)*rescale1[ibin-1])
            bdt_had.SetBinContent(ibin, bdt_had.GetBinContent(ibin)*rescale1[ibin-1])
        if len(rescale2) > 0:
            std_sim.SetBinContent(ibin, std_sim.GetBinContent(ibin)*rescale2[ibin-1])
            std_had.SetBinContent(ibin, std_had.GetBinContent(ibin)*rescale2[ibin-1])


    hists_pt_real[0][iED[0]][0][icentr].Draw("same")
    hists_pt_real[1][iED[1]][0][icentr].Draw("same")
    Format_Graph(bdt_sim, Lcolor = root.kBlue, Lwidth=3)
    Format_Graph(bdt_had, Lcolor = root.kBlue-7, Lwidth=3)
    Format_Graph(std_sim, Lcolor = root.kRed, Lwidth=3)
    Format_Graph(std_had, Lcolor = root.kRed-7, Lwidth=3)
    bdt_sim.Draw("same HIST")
    bdt_had.Draw("same HIST")
    std_sim.Draw("same HIST")
    std_had.Draw("same HIST")
    
    legends.append(root.TLegend(0.4,0.88,0.99,0.99,"0-20% Au+Au 200 GeV"))
    Format_Legend(legends[-1])
    legends[-1].Draw()
    legends.append(root.TLegend(0.4,0.84,0.99,0.92,""))
    legends[-1].SetNColumns(2)
    legends[-1].AddEntry(hists_pt_real[0][iED[0]][0][icentr],"data std","l")
    legends[-1].AddEntry(hists_pt_real[0][iED[1]][0][icentr],"data bdt","l")
    Format_Legend(legends[-1])
    legends[-1].Draw()
    legends.append(root.TLegend(0.4,0.78,0.99,0.86,""))
    legends[-1].SetNColumns(2)
    legends[-1].AddEntry(std_sim,"e sim std","l")
    legends[-1].AddEntry(bdt_sim,"e sim bdt","l")
    Format_Legend(legends[-1])
    legends[-1].Draw()
    legends.append(root.TLegend(0.4,0.72,0.99,0.80,""))
    legends[-1].SetNColumns(2)
    legends[-1].AddEntry(std_had,"h sim std","l")
    legends[-1].AddEntry(bdt_had,"h sim bdt","l")
    Format_Legend(legends[-1])
    legends[-1].Draw()
if not Save_to_Html: c0.Draw()
c0.SaveAs("output/electron_hadron_decomposition.png")

In [10]:
#stop

In [11]:
c0 = root.TCanvas(f"c0",f"c0",1800,1200)
c0.Divide(3,2)
h = []
legends = []
iED = [0,4]
for icentr in range(N_real_centr):
    c0.cd(icentr+1)
    h.append(root.TH1D(f"name{icentr}{ieID}{iconv}", f"name{icentr}{ieID}{iconv}", 100, 0.0, 4.4))
    h[-1].SetMaximum(1.2* max(hists_pt_real[0][iED[0]][0][icentr].GetMaximum(),hists_pt_real[0][iED[1]][0][icentr+5].GetMaximum()))
    h[-1].SetMinimum(10)
    Format_Hist_total(h[-1], title_x="p_{T} [GeV]",  title_y="dN/dp_{T}", left=0.15, bottom=0.15, right=0.01, top=0.01,  Tsize=0.07,  Lsize=0.06,\
                          Mstyle=21,  Msize=0, Mcolor=1,  Lwidth=6,  Lcolor=1,  offset_x=1, offset_y=1, title="",  Malpha=1,  Lalpha=1)
    
    Format_Hist_total(hists_pt_real[0][iED[0]][0][icentr], title_x="p_{T} [GeV]",  title_y="dN/dp_{T}", left=0.15, bottom=0.15, right=0.01, top=0.01,  Tsize=0.07,  Lsize=0.06,\
              Mstyle=21,  Msize=0, Mcolor=1,  Lwidth=6,  Lcolor=1,  offset_x=1, offset_y=1, title="",  Malpha=1,  Lalpha=1)
    Format_Hist_total(hists_pt_real[0][iED[1]][0][icentr+5], title_x="p_{T} [GeV]",  title_y="dN/dp_{T}", left=0.15, bottom=0.15, right=0.01, top=0.01,  Tsize=0.07,  Lsize=0.06,\
              Mstyle=21,  Msize=0, Mcolor=4,  Lwidth=4,  Lcolor=4,  offset_x=1, offset_y=1, title="",  Malpha=1,  Lalpha=1)
    root.gPad.SetLogy()
    h[-1].Draw()
    hists_pt_real[0][iED[0]][0][icentr].Draw("same")
    hists_pt_real[0][iED[1]][0][icentr+5].Draw("same")
    legends.append(root.TLegend(0.5,0.7,0.9,0.9))
    legends[-1].AddEntry(hists_pt_real[0][iED[0]][0][icentr],"std","l")
    legends[-1].AddEntry(hists_pt_real[0][iED[1]][0][icentr+5],"bdt","l")
    Format_Legend(legends[-1])
    legends[-1].Draw()
if not Save_to_Html: c0.Draw()

In [12]:
references = [0,4]

In [13]:
rebin = 1
for icentr in range(N_centr):
    for ieID in range(0,len(eIDbins)):
        for iconv in range(len(primaryID)):
            hists_pt_real[0][ieID][iconv][icentr].Rebin(rebin)
            hists_pt_sim[0][ieID][iconv][icentr].Rebin(rebin)
            hists_pt_sim[1][ieID][iconv][icentr].Rebin(rebin)

In [14]:
graphs= [] 
for reference in references:
    graphs_ref = []
    for icentr in range(N_centr):
        graphs_centr = []
        for ieID in range(0,len(eIDbins)):
            if ieID == reference:continue
            graphs_eid = []
            for iconv in range(len(primaryID)):
                graphs_conv = []
                for iptbin in range(1,1+hists_pt_real[0][ieID][iconv][icentr].GetNbinsX()):
                    raw_signal_frac =  hists_pt_real[0][ieID][iconv][icentr].GetBinContent(iptbin)/(hists_pt_real[0][reference][iconv][icentr].GetBinContent(iptbin)+1e-6)
                    raw_primary_frac = hists_pt_sim[0][ieID][iconv][icentr].GetBinContent(iptbin)/(hists_pt_sim[0][reference][iconv][icentr].GetBinContent(iptbin)+1e-6)
                    raw_hadron_frac =  hists_pt_sim[1][ieID][iconv][icentr].GetBinContent(iptbin)/(hists_pt_sim[1][reference][iconv][icentr].GetBinContent(iptbin)+1e-6)
                    #print(raw_signal_frac, raw_primary_frac, raw_hadron_frac)
                    if (raw_primary_frac - raw_signal_frac) == 0: hadron_frac = 0
                    else: 
                        if raw_primary_frac - raw_hadron_frac > 0:
                            hadron_frac = ( raw_primary_frac - raw_signal_frac ) / ( raw_primary_frac - raw_hadron_frac )
                        else: hadron_frac = -1
                        if hadron_frac<0: hadron_frac = 0
                        if hadron_frac>1: hadron_frac = 1
                    hadron_frac_err = hadron_frac *0.1+0.05 #(hists_pt_sim[1][ieID][iconv][icentr].GetBinError(iptbin)/(hists_pt_sim[1][ieID][iconv][icentr].GetBinContent(iptbin)+1e-6))*2**0.5
                        #print(hadron_frac)
                    graphs_conv.append([hists_pt_real[0][ieID][iconv][icentr].GetBinCenter(iptbin),hadron_frac,hadron_frac_err])
                graphs_eid.append(graphs_conv)
            graphs_centr.append(graphs_eid)
        graphs_ref.append(graphs_centr)
    graphs.append(graphs_ref)
if not Save_to_Html: print(graphs)

[[[[[[0.05, 0, 0.05], [0.15000000000000002, 1, 0.15000000000000002], [0.25, 1, 0.15000000000000002], [0.35000000000000003, 1, 0.15000000000000002], [0.45, 0.3975552867642481, 0.08975552867642482], [0.55, 0.4227917509535758, 0.09227917509535759], [0.6500000000000001, 0.44349776749590575, 0.09434977674959058], [0.7500000000000001, 0.43822360502208096, 0.0938223605022081], [0.8500000000000001, 0.4595902116022545, 0.09595902116022545], [0.9500000000000001, 0.5107693782579692, 0.10107693782579692], [1.05, 0.468311567202338, 0.0968311567202338], [1.1500000000000001, 0.45851229671630517, 0.09585122967163053], [1.2500000000000002, 0.4499375859985226, 0.09499375859985226], [1.35, 0.4585876371483894, 0.09585876371483895], [1.4500000000000002, 0.5152382506827632, 0.10152382506827633], [1.55, 0.4486521948437835, 0.09486521948437836], [1.6500000000000001, 0.41275295561927383, 0.09127529556192739], [1.7500000000000002, 0.3656938866978341, 0.08656938866978342], [1.85, 0.3610682973305834, 0.0861068297

In [15]:
Tgraphs= [] 
for reference in range(len(references)):
    graphs_ref = []
    for icentr in range(N_centr):
        graphs_centr = []
        for ieID in range(0,len(eIDbins)-1):
            graphs_eid = []
            for iconv in range(len(primaryID)):
                graphs_conv = root.TGraphErrors()
                ipoint = 0
                for iptbin in range(0,0+hists_pt_real[0][ieID][iconv][icentr].GetNbinsX()):
                    if hists_pt_real[0][ieID][iconv][icentr].GetBinCenter(iptbin+1)<0.4: continue
                    if hists_pt_real[0][ieID][iconv][icentr].GetBinCenter(iptbin+1)>4.4: continue
                    if graphs[reference][icentr][ieID][iconv][iptbin][1]<0.01: continue
                    graphs_conv.AddPoint(hists_pt_real[0][ieID][iconv][icentr].GetBinCenter(iptbin+1),graphs[reference][icentr][ieID][iconv][iptbin][1])
                    graphs_conv.SetPointError(ipoint,hists_pt_real[0][ieID][iconv][icentr].GetBinWidth(iptbin+1)/2,graphs[reference][icentr][ieID][iconv][iptbin][2])
                    ipoint+=1

                graphs_eid.append(graphs_conv)
            graphs_centr.append(graphs_eid)
        graphs_ref.append(graphs_centr)

    Tgraphs.append(graphs_ref)

In [16]:
only_pos = 1
c0 = root.TCanvas(f"c0",f"c0",1400,1400*(2-only_pos))
N_centr_loc = N_centr - 5*only_pos
c0.Divide(3,3*(2-only_pos))
h = []
TMultigraphs = []
for reference in range(len(references)):
    for icentr in range(N_centr_loc):
        graphs_centr = []
        for iconv in range(1,len(primaryID)):
            c0.cd(icentr+1+N_centr_loc*(iconv-1))
            TMultigraphs.append(root.TMultiGraph())
            for ieID in range(0,len(eIDbins)-1):
                graphs_eid = []
                if ieID == 0 :
                    h.append(root.TH1D(f"name{icentr}{ieID}{iconv}", f"name{icentr}{ieID}{iconv}", 100, 0.4, 4.9))
                    h[-1].SetMaximum(0.99)
                    Format_Hist_total(h[-1], title_x="p_{T} [GeV]",  title_y="dN/dp_{T}", left=0.15, bottom=0.15, right=0.01, top=0.01,  Tsize=0.07,  Lsize=0.06,\
                                          Mstyle=21,  Msize=0, Mcolor=1,  Lwidth=6,  Lcolor=1,  offset_x=1, offset_y=1, title="",  Malpha=1,  Lalpha=1)
                    h[-1].Draw()
                    Format_Graph( Tgraphs[reference][icentr][ieID][iconv],  Mstyle=21,  Msize=1,  Mcolor=1,  Lwidth=1,  Lcolor=1,  Malpha=1,  Lalpha=1)
                    Tgraphs[reference][icentr][ieID][iconv].Draw("same P")
                else :
                    Format_Graph( Tgraphs[reference][icentr][ieID][iconv],  Mstyle=20,  Msize=1,  Mcolor=2+ieID,  Lwidth=1,  Lcolor=1,  Malpha=2,  Lalpha=1)
                    Tgraphs[reference][icentr][ieID][iconv].Draw("same P")
                #TMultigraphs[-1].Add(Tgraphs[icentr][ieID][iconv])
                if ieID == len(eIDbins)-4 or True: Tgraphs[reference][icentr][ieID][iconv].Fit("pol0","Q")
                #TMultigraphs[-1].Draw()
if not Save_to_Html: c0.Draw()

In [17]:
hists_pt_real, hists_pt_sim = [], []

for iFilereal in range(len(real_file_names)):
    hists_pt_real_eid = []
    for ieID in range(len(eIDbins)):
        hists_pt_real_conv = []
        for iconv in range(len(primaryID)):
            hists_pt_real_centr = []
            for icentr in range(N_centr):
                hists_pt_real_centr.append(hists_read_real[0][icentr].ProjectionX(hists_read_real[iFilereal][icentr].GetName()+f"_pr{ieID}_{iconv}_0",eIDbins[ieID],eIDbins[ieID],primaryID[iconv][0],primaryID[iconv][0]))
                for jconv in range(1,len(primaryID[iconv])):
                    hists_pt_real_centr[-1].Add(hists_read_real[0][icentr].ProjectionX(hists_read_real[iFilereal][icentr].GetName()+f"_pr{ieID}_{iconv}_{jconv}",eIDbins[ieID],eIDbins[ieID],primaryID[iconv][jconv],primaryID[iconv][jconv]))
            hists_pt_real_conv.append(hists_pt_real_centr)
        hists_pt_real_eid.append(hists_pt_real_conv)
    hists_pt_real.append(hists_pt_real_eid)
    

for iFilesim in range(len(sim_file_names)):
    hists_pt_sim_eid = []
    for ieID in range(len(eIDbins)):
        hists_pt_sim_conv = []
        for iconv in range(len(primaryID)):
            hists_pt_sim_centr = []
            for icentr in range(N_centr):
                hists_pt_sim_centr.append(hists_read_sim[iFilesim][icentr].ProjectionX(hists_read_sim[iFilesim][icentr].GetName()+f"_pr{ieID}_{iconv}_0",eIDbins[ieID],eIDbins[ieID],primaryID[iconv][0],primaryID[iconv][0]))
                for jconv in range(1,len(primaryID[iconv])):
                    hists_pt_sim_centr[-1].Add(hists_read_sim[iFilesim][icentr].ProjectionX(hists_read_sim[iFilesim][icentr].GetName()+f"_pr{ieID}_{iconv}_{jconv}",eIDbins[ieID],eIDbins[ieID],primaryID[iconv][jconv],primaryID[iconv][jconv]))
            hists_pt_sim_conv.append(hists_pt_sim_centr)
        hists_pt_sim_eid.append(hists_pt_sim_conv)
    hists_pt_sim.append(hists_pt_sim_eid)

In [18]:
graphs= [] 
for reference in references:
    graphs_ref = []
    for icentr in range(N_centr):
        graphs_centr = []
        for ieID in range(1,len(eIDbins)):
            graphs_eid = []
            for iconv in range(len(primaryID)):
                raw_signal_frac =  hists_pt_real[0][ieID][iconv][icentr].Integral(5,10)/(hists_pt_real[0][reference][iconv][icentr].Integral(5,10)+1e-6)
                raw_primary_frac = hists_pt_sim[0][ieID][iconv][icentr].Integral(5,10)/(hists_pt_sim[0][reference][iconv][icentr].Integral(5,10)+1e-6)
                raw_hadron_frac =  hists_pt_sim[1][ieID][iconv][icentr].Integral(5,10)/(hists_pt_sim[1][reference][iconv][icentr].Integral(5,10)+1e-6)
                print(raw_signal_frac, raw_primary_frac, raw_hadron_frac)
                if (raw_signal_frac - raw_primary_frac) == 0: hadron_frac = 0
                else: 
                    if raw_primary_frac - raw_hadron_frac > 0:
                        hadron_frac = ( raw_primary_frac - raw_signal_frac ) / ( raw_primary_frac - raw_hadron_frac )
                    else: hadron_frac = -1
                    if hadron_frac<0: hadron_frac = 0
                    if hadron_frac>1: hadron_frac = 1
                graphs_eid.append(hadron_frac)
            graphs_centr.append(graphs_eid)
        graphs_ref.append(graphs_centr)
    graphs.append(graphs_ref)
if not Save_to_Html: print(graphs)

0.365145048786718 0.5582611043918735 0.09801217969688472
0.30365974121008693 0.5795340681241747 0.09685414678801636
0.3040535107647157 0.5826019512355552 0.10170025185714728
0.2539063401675083 0.5364638255154416 0.048029415138684424
0.21068380891398855 0.5894497327866264 0.04690181123986619
0.21042522106136954 0.5927256398500744 0.05384130980672503
0.20155516484637517 0.4485545639229408 0.03125359071225398
0.16753655560745678 0.4947812291145879 0.0301239275443043
0.1676278149696764 0.49924480546600625 0.03243073046837824
0.1356265739830413 0.31580334867727294 0.01206480523818628
0.11286104055946541 0.3419964929788327 0.012392755002403668
0.1131949604375466 0.3469812629976331 0.013224181356037727
0.48317423835444795 0.6074223060055143 0.10434007131262187
0.42252074681756285 0.6186528980790554 0.10381275949270942
0.4162997208045787 0.6199407195262688 0.10304568522688036
0.4466912358171759 0.713000384603614 0.08263971460087999
0.3904858001747986 0.7534564003553061 0.0856927141994365
0.384

In [19]:
if True :
    only_pos = 1
    c0 = root.TCanvas(f"c0",f"c0",1400,500*(2-only_pos))
    N_centr_loc = N_centr - 5*only_pos
    c0.Divide(3,1*(2-only_pos))
    legends = []
    for reference, iref in zip(references, range(len(references))):
        for icentr in range(3):
            for iconv in range(1):
                c0.cd(icentr+1+N_centr_loc*iconv)
                root.gPad.SetLogy()

                Format_Hist_total(hists_pt_real[0][reference][iconv][icentr], title_x="p_{T} [GeV]",  title_y="dN/dp_{T}", left=0.15, bottom=0.15, right=0.01, top=0.01,  Tsize=0.07,  Lsize=0.06,\
                          Mstyle=21,  Msize=0, Mcolor=2,  Lwidth=4,  Lcolor=2,  offset_x=1, offset_y=1.1, title="",  Malpha=1,  Lalpha=1)

                hists_pt_real[0][reference][iconv][icentr].GetXaxis().SetRange(4,45)
                if reference == 0: hists_pt_real[0][reference][iconv][icentr].Draw("H")
                else: hists_pt_real[0][reference][iconv][icentr].Draw("same H")

                for iptbin in range(1,1+hists_pt_real[0][0][iconv][icentr].GetNbinsX()):
                    kek = 0
                    if  Tgraphs[iref][icentr][1][1].GetFunction("pol0") and icentr<5:
                        kek = Tgraphs[iref][icentr][1][1].GetFunction("pol0").Eval(hists_pt_real[0][reference][iconv][icentr].GetBinCenter(iptbin))/1.5
                    
                    hists_pt_sim[1][0][iconv][icentr].SetBinContent(iptbin,kek*hists_pt_real[0][reference][iconv][icentr].GetBinContent(iptbin))
                    #hists_pt_sim[1][1][iconv][icentr].SetBinContent(iptbin,Tgraphs[icentr][3][1].GetFunction("pol2").Eval(hists_pt_real[0][0][iconv][icentr].GetBinCenter(iptbin))*hists_pt_real[0][0][iconv][icentr].GetBinContent(iptbin))

                Format_Hist_total(hists_pt_sim[1][0][iconv][icentr], title_x="p_{T} [GeV]",  title_y="dN/dp_{T}", left=0.15, bottom=0.15, right=0.01, top=0.01,  Tsize=0.07,  Lsize=0.06,\
                          Mstyle=21,  Msize=0, Mcolor=1,  Lwidth=4,  Lcolor=1,  offset_x=1, offset_y=1.1, title="",  Malpha=1,  Lalpha=1)
                Format_Hist_total(hists_pt_sim[1][1][iconv][icentr], title_x="p_{T} [GeV]",  title_y="dN/dp_{T}", left=0.15, bottom=0.15, right=0.01, top=0.01,  Tsize=0.07,  Lsize=0.06,\
                          Mstyle=21,  Msize=0, Mcolor=4,  Lwidth=4,  Lcolor=4,  offset_x=1, offset_y=1.1, title="",  Malpha=1,  Lalpha=1)
                hists_pt_sim[1][0][iconv][icentr].Draw('same H')
                legends.append(root.TLegend(0.4,0.68,0.9,0.98, f"Au+Au {central_bins[icentr*2]}-{central_bins[icentr*2+1]}%"))
                legends[-1].AddEntry(hists_pt_real[0][reference][iconv][icentr],"BDT basic","l")
                legends[-1].AddEntry(hists_pt_sim[1][0][iconv][icentr],"hadrons","l")
                Format_Legend(legends[-1])
                legends[-1].Draw()
                #hists_pt_sim[1][1][iconv][icentr].Draw('same H')
c0.Draw()         
c0.SaveAs("output/BDT_basic_hadron_fraction.png")   

In [20]:
if False:
    outfilename = f"output/files/sngl_spectra_{reference}.root"
    outfile = root.TFile.Open(outfilename, "recreate")
    for icentr in range(N_centr_loc):
        hists_pt_real[0][reference][0][icentr].Write()
        hists_pt_sim[1][0][0][icentr].Write()
    outfile.Close()

In [21]:
if True :
    best_type = 2
    only_pos = 1
    c0 = root.TCanvas(f"c0",f"c0",1400,1000*(2-only_pos))
    N_centr_loc = N_centr - 5*only_pos
    c0.Divide(3,2*(2-only_pos))
    hist_pt_orig_centr = []
    legends=[]
    for icentr in range(N_centr_loc):
        hist_pt_orig_centr.append(hist_pt_orig.ProjectionX(f"orig{icentr}",icentr+1,icentr+1,1,1))
        for iconv in range(1):
            c0.cd(icentr+1+N_centr_loc*iconv)
            #root.gPad.SetLogy()
                
            Format_Hist_total(hists_pt_real[0][best_type][iconv][icentr], title_x="p_{T} [GeV]",  title_y="1/2#pip_{T} d^{2}N/dydp_{T}", left=0.3, bottom=0.15, right=0.01, top=0.01,  Tsize=0.07,  Lsize=0.06,\
                      Mstyle=21,  Msize=1, Mcolor=2,  Lwidth=4,  Lcolor=2,  offset_x=1, offset_y=1.5, title="",  Malpha=1,  Lalpha=1)
            
            hists_pt_real[0][best_type][iconv][icentr].GetXaxis().SetRange(4,45)

            for iptbin in range(1,1+hists_pt_real[0][0][iconv][icentr].GetNbinsX()):
                bincontent = hists_pt_real[0][best_type][iconv][icentr].GetBinContent(iptbin)/(hists_pt_sim[0][best_type][iconv][icentr].GetBinContent(iptbin)+1e-12)*hist_pt_orig_centr[icentr].GetBinContent(iptbin)
                error = hists_pt_real[0][best_type][iconv][icentr].GetBinContent(iptbin)**0.5/(hists_pt_sim[0][best_type][iconv][icentr].GetBinContent(iptbin)+1e-12)*hist_pt_orig_centr[icentr].GetBinContent(iptbin)
                bincontent /= 2*3.14*hists_pt_real[0][best_type][iconv][icentr].GetBinWidth(iptbin)*hists_pt_real[0][best_type][iconv][icentr].GetBinCenter(iptbin)
                error /= 3.14*hists_pt_real[0][best_type][iconv][icentr].GetBinWidth(iptbin)*hists_pt_real[0][best_type][iconv][icentr].GetBinCenter(iptbin)
                hists_pt_real[0][best_type][iconv][icentr].SetBinContent(iptbin,bincontent)
                hists_pt_real[0][best_type][iconv][icentr].SetBinError(iptbin,error)

                bincontent = hists_pt_real[0][best_type+1][iconv][icentr].GetBinContent(iptbin)/(hists_pt_sim[0][best_type+1][iconv][icentr].GetBinContent(iptbin)+1e-12)*hist_pt_orig_centr[icentr].GetBinContent(iptbin)
                error = hists_pt_real[0][best_type+1][iconv][icentr].GetBinContent(iptbin)**0.5/(hists_pt_sim[0][best_type+1][iconv][icentr].GetBinContent(iptbin)+1e-12)*hist_pt_orig_centr[icentr].GetBinContent(iptbin)
                bincontent /= 2*3.14*hists_pt_real[0][best_type][iconv][icentr].GetBinWidth(iptbin)*hists_pt_real[0][best_type][iconv][icentr].GetBinCenter(iptbin)
                error /= 3.14*hists_pt_real[0][best_type][iconv][icentr].GetBinWidth(iptbin)*hists_pt_real[0][best_type][iconv][icentr].GetBinCenter(iptbin)
                hists_pt_real[0][best_type+1][iconv][icentr].SetBinContent(iptbin,bincontent)
                hists_pt_real[0][best_type+1][iconv][icentr].SetBinError(iptbin,error)
                
            
            hists_pt_real[0][best_type][iconv][icentr].Scale(1./2.413e9)
            hists_pt_real[0][best_type+1][iconv][icentr].Scale(1./2.413e9)
            Format_Hist_total(hists_pt_real[0][best_type+1][iconv][icentr], title_x="p_{T} [GeV]",  title_y="dN/dp_{T}", left=0.21, bottom=0.15, right=0.01, top=0.01,  Tsize=0.07,  Lsize=0.06,\
                      Mstyle=20,  Msize=1, Mcolor=4,  Lwidth=4,  Lcolor=4,  offset_x=1, offset_y=1, title="",  Malpha=1,  Lalpha=1)
            
            legends.append(root.TLegend(0.4,0.7,0.9,0.98, "e^{+} "+f"Au+Au {central_bins[icentr*2]}-{central_bins[icentr*2+1]}%"))
            legends[-1].AddEntry(hists_pt_real[0][best_type][iconv][icentr],"bdt loose","l")
            legends[-1].AddEntry(hists_pt_real[0][best_type+1][iconv][icentr],"bdt tight","l")
            Format_Legend(legends[-1])
            hists_pt_real[0][best_type][iconv][icentr].Divide(hists_pt_real[0][best_type+1][iconv][icentr])
            hists_pt_real[0][best_type][iconv][icentr].Draw()
            #hists_pt_real[0][best_type+1][iconv][icentr].Draw("same")

            legends[-1].Draw()
    c0.Draw()  
    #c0.SaveAs("output/ptspectrum__loose_tight.png")          

In [22]:
if False:
    outfilename = f"output/files/reco_spectra.root"
    outfile = root.TFile.Open(outfilename, "recreate")
    for icentr in range(N_centr_loc):
        hists_pt_real[0][best_type][0][icentr].Write()
        hists_pt_real[0][best_type+1][0][icentr].Write()
    outfile.Close()